<!-- Copyright 2026 The cuju Authors. SPDX-License-Identifier: Apache-2.0 -->
# cuju quickstart

cuju is a bit-exact, batched port of Google Research Football, or GRF, for CUDA and the CPU. It plays
thousands of GRF matches at once, and for the same seed and actions every observation, reward and
done flag equals what GRF gives, bit for bit.

In about ten minutes on Colab's free T4, this notebook measures the speed of a batch, shows the CPU
and the GPU agreeing to the last bit, draws a match inline, trains a policy that learns to score,
and runs that policy through GRF's own interface. Pick a GPU runtime first, under Runtime, then
Change runtime type, then T4 GPU, and run the cells in order.

## 1. Install and check

Colab already has PyTorch, so pip adds cuju and, through the `spaces` extra, gymnasium, which GRF's
interface in section 6 uses. The commented line installs a wheel file instead. The self-check then
plays short matches on the GPU and compares every output with stored fingerprints of GRF's matches.
It ends with PASSED when this runtime reproduces them exactly.

In [ ]:
%pip install -q 'cuju[spaces]'
# %pip install -q './cuju-0.1.0-<tags>.whl[spaces]'

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python -m cuju.check --device cuda

## 2. Speed

Training on GRF spends most of its time waiting for the simulator, so throughput comes first. The
cell times batches of 256 to 16,384 five-a-side matches, with four agents on each side and random
actions, for about ten seconds each. The dashed line is one GRF process, which plays about 200 steps
per second on a modern CPU. A size that would overflow the free memory of the GPU is left out.

In [ ]:
import time

import matplotlib.pyplot as plt
import torch

import cuju


def steps_per_second(count, seconds=10):
    with cuju.BatchEnvironment('5_vs_5_l4_r4', environment_count=count, device='cuda') as batch:
        batch.reset()
        actions = torch.zeros((count, batch.agents), dtype=torch.int32, device='cuda')
        for _ in range(2):
            batch.step(actions.random_(0, batch.action_count))
        torch.cuda.synchronize()
        start, steps = time.perf_counter(), 0
        while time.perf_counter() - start < seconds:
            batch.step(actions.random_(0, batch.action_count))
            torch.cuda.synchronize()
            steps += 1
        return steps * count / (time.perf_counter() - start)


rates = {}
for count in (256, 1024, 4096, 16384):
    if count * 0.4 * 2**20 > 0.8 * torch.cuda.mem_get_info()[0]:
        print(f'{count:>6,} environments need more memory than the GPU has free')
        continue
    rates[count] = steps_per_second(count)
    print(f'{count:>6,} environments  {rates[count]:>8,.0f} steps per second')

plt.figure(figsize=(6, 3.5))
plt.plot(list(rates), list(rates.values()), 'o-', label=f'cuju on {torch.cuda.get_device_name()}')
plt.axhline(207, color='gray', linestyle='--', label='one GRF process')
plt.xscale('log', base=2)
plt.xlabel('environments in the batch')
plt.ylabel('environment steps per second')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

On a T4 the batch reaches about 3,800 steps per second, nearly twenty GRF processes, and an RTX 4090
reaches about 32,000. The observations, the actions and the policy all stay in GPU memory, so a
training loop copies nothing to the host. [Performance](https://github.com/AndreKoraleski/cuju/blob/main/docs/performance.md) has the method and
figures for more GPUs and CPUs.

## 3. The same bits on the CPU and the GPU

Both backends compile the same C++ source, and both must give GRF's bits. The cell plays 64
environments with the same seed and the same random actions on each device for 300 steps, and
compares every output with `torch.equal` after every step. Half the batch plays five-a-side, and the
other half plays `academy_3_vs_1_with_keeper`, a drill that ends every few dozen steps, so the
resets and the final observations of ended episodes are compared too.

In [ ]:
scenarios = ['5_vs_5_l4_r0', 'academy_3_vs_1_with_keeper_l3_r0']
options = dict(environment_count=64, seed=7, assignment=[index % 2 for index in range(64)])
cpu = cuju.BatchEnvironment(scenarios, device='cpu', **options)
gpu = cuju.BatchEnvironment(scenarios, device='cuda', **options)


def outputs(observation, reward, done, information):
    return {'observation': observation, 'reward': reward, 'done': done, **information}


differences = [] if torch.equal(cpu.reset(), gpu.reset().cpu()) else ['reset']
generator = torch.Generator().manual_seed(1)
episodes = 0
for step in range(300):
    actions = torch.randint(0, 19, (64, cpu.agents), dtype=torch.int32, generator=generator)
    on_cpu = outputs(*cpu.step(actions))
    on_gpu = outputs(*gpu.step(actions.cuda()))
    differences += [
        f'{name} at step {step}'
        for name in on_cpu
        if not torch.equal(on_cpu[name], on_gpu[name].cpu())
    ]
    episodes += int(on_cpu['done'].sum())

print(f'compared {", ".join(on_cpu)} after each of 300 steps, {episodes} episodes ended')
print('every tensor equal on both devices' if not differences else differences[:5])
cpu.close()
gpu.close()

## 4. A match on the board

The board view draws the pitch, both teams, the ball and the score from the `raw` observation, the
dictionary of GRF's arrays that `with_raw=True` adds to a batch. Its code is plain matplotlib, folded
into the next cell.

In [ ]:
# @title Board view { display-mode: "form" }
import numpy as np
from IPython.display import HTML
from matplotlib import animation
from matplotlib.patches import Arc, Circle, Rectangle

X_SCALE, Y_SCALE = 54.4, 83.6
FIELDS = (
    'ball',
    'left_team',
    'right_team',
    'left_team_active',
    'right_team_active',
    'left_team_roles',
    'right_team_roles',
    'score',
    'game_mode',
    'active',
)
KITS = {'left_team': ('#3b82f6', '#facc15'), 'right_team': ('#ef4444', '#a3e635')}
MODES = ('', 'kick-off', 'goal kick', 'free kick', 'corner', 'throw-in', 'penalty')
plt.rcParams['animation.embed_limit'] = 64


def capture(raw):
    """The board of every environment after a step, as agent 0 sees it, kept on the device."""
    return {name: raw[name][:, 0].clone() for name in FIELDS}


def replay(frames, environment):
    """The frames of one environment as numpy arrays, time first."""
    return {
        name: torch.stack([frame[name][environment] for frame in frames]).cpu().numpy()
        for name in FIELDS
    }


def draw_pitch(ax):
    for index in range(14):
        colour = ('#2a7a48', '#256f41')[index % 2]
        ax.add_patch(Rectangle((-55 + index * 110 / 14, -36), 110 / 14, 72, color=colour))
    line = dict(fill=False, color='white', linewidth=1.3)
    ax.add_patch(Rectangle((-55, -36), 110, 72, **line))
    ax.plot([0, 0], [-36, 36], color='white', linewidth=1.3)
    ax.add_patch(Circle((0, 0), 9.15, **line))
    for side in (-1, 1):
        goal_line = side * 55
        ax.add_patch(Rectangle((goal_line - (side > 0) * 16.5, -20.16), 16.5, 40.32, **line))
        ax.add_patch(Rectangle((goal_line - (side > 0) * 5.5, -9.16), 5.5, 18.32, **line))
        ax.add_patch(Rectangle((goal_line - (side < 0) * 2.4, -3.66), 2.4, 7.32, **line))
        centre = 0 if side < 0 else 180
        arc = Arc((side * 44, 0), 18.3, 18.3, theta1=centre - 53, theta2=centre + 53, **line)
        ax.add_patch(arc)
    ax.set_xlim(-59, 59)
    ax.set_ylim(39, -39)
    ax.set_aspect('equal')
    ax.axis('off')


def animate(match, title, first=0, fps=10):
    """An inline animation of the frames of `replay`, at GRF's ten steps per second."""
    fig, ax = plt.subplots(figsize=(7.2, 5.2), dpi=80)
    fig.patch.set_facecolor('#0d131a')
    fig.subplots_adjust(0.01, 0.01, 0.99, 0.88)
    draw_pitch(ax)
    teams = {key: ax.scatter([], [], s=110, edgecolors='white', zorder=3) for key in KITS}
    ring = ax.scatter([], [], s=330, facecolors='none', edgecolors='white', zorder=3)
    shadow = ax.scatter([], [], s=30, color='black', alpha=0.4, zorder=4)
    ball = ax.scatter([], [], s=55, color='white', edgecolors='black', zorder=5)
    text = dict(color='white', va='center')
    score = fig.text(0.5, 0.94, '', ha='center', fontsize=18, weight='bold', **text)
    clock = fig.text(0.03, 0.94, '', fontsize=11, **text)
    fig.text(0.97, 0.94, title, ha='right', fontsize=11, **text)
    for x, key, name in ((0.37, 'left_team', 'LEFT'), (0.63, 'right_team', 'RIGHT')):
        fig.text(x, 0.94, name, color=KITS[key][0], ha='center', va='center', weight='bold')
    banner = ax.text(0, -22, '', ha='center', fontsize=36, weight='bold', zorder=6, **text)
    goals = np.flatnonzero(np.diff(match['score'].sum(1)) > 0) + 1

    def update(frame):
        for key, (kit, keeper) in KITS.items():
            present = match[key + '_active'][frame]
            teams[key].set_offsets(match[key][frame][present] * (X_SCALE, Y_SCALE))
            roles = match[key + '_roles'][frame][present]
            teams[key].set_facecolor([keeper if role == 0 else kit for role in roles])
        active = match['active'][frame]
        ring.set_offsets(match['left_team'][frame][[active]] * (X_SCALE, Y_SCALE))
        x, y, z = match['ball'][frame]
        shadow.set_offsets([(x * X_SCALE, y * Y_SCALE)])
        ball.set_offsets([(x * X_SCALE, y * Y_SCALE - 0.5 * z)])
        ball.set_sizes([55 * (1 + 0.1 * min(z, 10))])
        score.set_text('{}  -  {}'.format(*match['score'][frame]))
        clock.set_text(f'step {first + frame}   {MODES[match["game_mode"][frame]]}')
        banner.set_text('GOAL' if any(0 <= frame - goal < 15 for goal in goals) else '')

    movie = animation.FuncAnimation(fig, update, frames=len(match['ball']), interval=1000 / fps)
    plt.close(fig)
    return HTML(movie.to_jshtml(default_mode='loop'))

Here the built-in AI plays both teams of 256 five-a-side matches at once. The left team has one
agent, whose action 19 in the `v2` action set, `builtin_ai`, hands its player to the AI for that
step. The batch keeps the last 170 boards of every environment on the GPU and stops at the first
goal anywhere, then plays 20 more steps, so the board shows the build-up, the goal and the restart.
The seed fixes the matches, so every GPU, and the CPU, show this same goal.

In [ ]:
import collections

count = 256
batch = cuju.BatchEnvironment(
    '5_vs_5_l1_r0', environment_count=count, device='cuda', action_set='v2', with_raw=True, seed=3
)
builtin_ai = torch.full((count, 1), 19, dtype=torch.int32, device='cuda')
batch.reset()
frames = collections.deque([capture(batch.raw())], maxlen=170)
steps, scorers = 0, []
while not len(scorers):
    information = batch.step(builtin_ai)[3]
    frames.append(capture(batch.raw()))
    scorers = information['score_reward'].nonzero()
    steps += 1
for _ in range(20):
    batch.step(builtin_ai)
    frames.append(capture(batch.raw()))
batch.close()

environment = int(scorers[0])
print(f'environment {environment} scored first, at step {steps}')
animate(replay(frames, environment), '5_vs_5, built-in AI', first=steps + 21 - len(frames))

## 5. Training a policy that scores

Speed matters because on-policy methods such as PPO learn from fresh experience, and lots of it.
The cell below is a complete PPO in plain PyTorch, with the batch, the rollouts, the network and the
updates all on the GPU. It trains one agent on `academy_empty_goal_close`, GRF's drill of a striker with the ball
in front of an empty goal, with the batch's own rewards, the goal plus GRF's checkpoint reward for
carrying the ball toward the goal, which `rewards='scoring,checkpoints'` turns on.

The goal rate is the share of finished episodes that end in a goal, measured on the episodes of each
rollout. It starts low for the random policy and passes 0.9 after about a million environment steps,
under a minute on an RTX 4090 and about four minutes on a T4.

In [ ]:
from torch import nn

scenario, count, horizon, budget = 'academy_empty_goal_close', 2048, 16, 1_200_000
gamma, lam, epochs, minibatches = 0.993, 0.95, 4, 4
batch = cuju.BatchEnvironment(
    scenario, environment_count=count, device='cuda', rewards='scoring,checkpoints', seed=1
)
device, agents = batch.torch_device, batch.agents
torch.manual_seed(0)


class Policy(nn.Module):
    def __init__(self, actions):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(115, 256), nn.Tanh(), nn.Linear(256, 256), nn.Tanh())
        self.logits = nn.Linear(256, actions)
        self.value = nn.Linear(256, 1)

    def forward(self, observation):
        hidden = self.body(observation)
        return self.logits(hidden), self.value(hidden).squeeze(-1)


policy = Policy(batch.action_count).to(device)
optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4, eps=1e-5)
shape = (horizon, count, agents)
observations = torch.empty((*shape, 115), device=device)
actions = torch.empty(shape, dtype=torch.long, device=device)
log_probs, values, rewards, dones = (torch.empty(shape, device=device) for _ in range(4))

observation = batch.reset()
steps, start, history = 0, time.perf_counter(), []
while steps < budget:
    goals = torch.zeros((), device=device)
    episodes = torch.zeros((), device=device)
    with torch.no_grad():
        for t in range(horizon):
            observations[t] = observation
            logits, values[t] = policy(observation)
            distribution = torch.distributions.Categorical(logits=logits)
            actions[t] = distribution.sample()
            log_probs[t] = distribution.log_prob(actions[t])
            observation, reward, done, information = batch.step(actions[t].to(torch.int32))
            rewards[t] = reward
            dones[t] = done[:, None]
            goals += (information['score_reward'] == 1).sum()
            episodes += done.sum()
        next_value = policy(observation)[1]
        advantages = torch.empty_like(rewards)
        gae = torch.zeros_like(next_value)
        for t in reversed(range(horizon)):
            alive = 1 - dones[t]
            delta = rewards[t] + gamma * next_value * alive - values[t]
            gae = delta + gamma * lam * alive * gae
            advantages[t] = gae
            next_value = values[t]
        returns = advantages + values
    flat = [x.flatten(0, 2) for x in (observations, actions, log_probs, advantages, returns)]
    size = flat[1].numel()
    for _ in range(epochs):
        for index in torch.randperm(size, device=device).split(size // minibatches):
            o, a, old, advantage, target = (x[index] for x in flat)
            logits, value = policy(o)
            distribution = torch.distributions.Categorical(logits=logits)
            ratio = (distribution.log_prob(a) - old).exp()
            advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-8)
            clipped = torch.min(ratio * advantage, ratio.clamp(0.8, 1.2) * advantage)
            entropy = distribution.entropy().mean()
            loss = -clipped.mean() + 0.5 * (value - target).pow(2).mean() - 0.01 * entropy
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
            optimizer.step()
    steps += horizon * count
    history.append((steps, float(goals / episodes.clamp(min=1))))
    if len(history) % 8 == 0:
        elapsed = time.perf_counter() - start
        print(f'{steps:>10,} steps  {elapsed:5.0f} s  goal rate {history[-1][1]:.2f}')
batch.close()

plt.figure(figsize=(6, 3.5))
plt.plot(*zip(*history, strict=True))
plt.xlabel('environment steps')
plt.ylabel('goal rate')
plt.title(scenario)
plt.grid(alpha=0.3)
plt.show()

Now the trained policy plays 512 fresh episodes with another seed, each environment exactly one
episode thanks to `auto_reset=False`, and the board replays one of its goals.

In [ ]:
def act(observation):
    with torch.no_grad():
        logits = policy(observation)[0]
    return torch.distributions.Categorical(logits=logits).sample().to(torch.int32)


count = 512
evaluation = cuju.BatchEnvironment(
    scenario, environment_count=count, device='cuda', with_raw=True, auto_reset=False, seed=12345
)
observation = evaluation.reset()
frames = [capture(evaluation.raw())]
finished = torch.zeros(count, dtype=torch.bool, device=device)
scored = torch.zeros(count, dtype=torch.bool, device=device)
lengths = torch.zeros(count, dtype=torch.int64, device=device)
while not finished.all():
    observation, _, done, information = evaluation.step(act(observation))
    frames.append(capture(evaluation.raw()))
    lengths += ~finished
    scored |= (information['score_reward'] == 1) & ~finished
    finished |= done.bool()
evaluation.close()

print(f'{int(scored.sum())} goals in {count} episodes')
environment = int(scored.nonzero()[0]) if scored.any() else 0
animate(replay(frames[: int(lengths[environment]) + 15], environment), scenario)

The striker has learned to take the ball forward and shoot. Harder drills run with the same code and
need far longer runs, and with these settings `academy_3_vs_1_with_keeper` reaches a goal rate of
about 0.3 after 16 million steps. [Recipes](https://github.com/AndreKoraleski/cuju/blob/main/docs/recipes.md) covers rollout buffers, bootstrapping,
self-play and evaluation against the built-in AI.

## 6. GRF code, unchanged

`cuju.gfootball` mirrors GRF's import paths, so code written for GRF runs with its imports switched,
including the old gym interface where `reset` returns the observation and `step` returns four values.
Here GRF's own loop plays the trained policy for 20 episodes on the CPU, one match per call, just as
GRF does. Swapping `cuju.gfootball` for `gfootball` runs the same cell on GRF.

In [ ]:
import cuju.gfootball.env as football_env

env = football_env.create_environment(env_name=scenario, representation='simple115v2')
goals = 0
for _ in range(20):
    observation, done = env.reset(), False
    while not done:
        action = int(act(torch.as_tensor(observation, device=device)))
        observation, reward, done, info = env.step(action)
    goals += info['score_reward']
env.close()
print(f'{goals} goals in 20 episodes')

## 7. Dumps for GRF's tools

GRF's replay and video tools read dump files, and `cuju.dumps.DumpWriter` writes them from a batch.
It follows the environments it is given and writes each episode as it ends. One episode of the
trained policy takes about a second, and `read_dump` reads the file back without GRF.

In [ ]:
import cuju.dumps

small = cuju.BatchEnvironment(scenario, environment_count=1, device='cuda', with_raw=True, seed=5)
writer = cuju.dumps.DumpWriter(small, 'dumps')
observation = small.reset()
while not writer.paths:
    action = act(observation)
    observation = small.step(action)[0]
    writer.record(action)
small.close()

dump = cuju.dumps.read_dump(writer.paths[0])
print(writer.paths[0], len(dump), 'steps, final score', dump[-1]['observation']['score'])

On a machine with GRF and its renderer, GRF's own tool turns the dump into a video, given the path
that the cell printed.

```sh
python -m gfootball.dump_to_video --trace_file=dumps/episode_done_environment_0_episode_1.dump
```

## 8. Next steps

The documentation goes deeper on every part of this notebook.

- [Concepts](https://github.com/AndreKoraleski/cuju/blob/main/docs/concepts.md), for batches, bundles and environments
- [Scenarios](https://github.com/AndreKoraleski/cuju/blob/main/docs/scenarios.md), for the nineteen GRF scenarios and agent counts
- [Observations](https://github.com/AndreKoraleski/cuju/blob/main/docs/observations.md) and [Rewards and actions](https://github.com/AndreKoraleski/cuju/blob/main/docs/rewards-and-actions.md)
- [Episodes and seeds](https://github.com/AndreKoraleski/cuju/blob/main/docs/episodes-and-seeds.md), for resets and reproducibility
- [Migrating from GRF](https://github.com/AndreKoraleski/cuju/blob/main/docs/migrating-from-grf.md), for every GRF name and its counterpart
- [Exactness](https://github.com/AndreKoraleski/cuju/blob/main/docs/exactness.md), for what equals GRF and how to check it
- [API reference](https://github.com/AndreKoraleski/cuju/blob/main/docs/api.md), for every public name

Issues are welcome on [GitHub](https://github.com/AndreKoraleski/cuju/issues), and a difference
from GRF is the most valued report of all.